# Auditoría del universo poblacional del ABM — 2023T4

**Objetivo:** identificar exactamente qué población está representando el ABM, qué filtros explican la diferencia frente al total EPH 2023T4 y por qué no coincide con los ~30 millones que reporta INDEC para los 31 aglomerados urbanos.

**Fuentes:**
- EPH individual 2023T4: `Datos/Bases EPH/bases_eph_TFM.zip` → `usu_individual_T423.txt`
- Población inicial del ABM: `Salidas/13_abm_base/poblacion_inicial_2023T4.csv`

In [1]:
import zipfile
import io
import pandas as pd
import numpy as np

RAIZ = r"c:\Users\tomas\OneDrive\Master Granada\TFM"
ZIP = RAIZ + r"\Datos\Bases EPH\bases_eph_TFM.zip"
ABM_CSV = RAIZ + r"\Salidas\13_abm_base\poblacion_inicial_2023T4.csv"

def cargar_eph(zip_path, archivo):
    with zipfile.ZipFile(zip_path) as z:
        with z.open(archivo) as f:
            return pd.read_csv(
                io.TextIOWrapper(f, encoding="latin1"),
                sep=";",
                dtype=str,
                low_memory=False,
            )

## 1. EPH completa 2023T4 — registros y población ponderada

In [2]:
eph = cargar_eph(ZIP, "usu_individual_T423.txt")

# Variables clave como numéricas
for c in ["PONDERA", "ESTADO", "EMPLEO", "CH06", "ANO4", "TRIMESTRE"]:
    eph[c] = pd.to_numeric(eph[c], errors="coerce")

total_reg = len(eph)
total_pob = eph["PONDERA"].sum()

print(f"Registros EPH 2023T4 : {total_reg:>9,}")
print(f"Suma PONDERA         : {total_pob:>12,.0f}")
print(f"Año/trimestre        : {eph['ANO4'].unique()}, T{eph['TRIMESTRE'].unique()}")
print(f"Ponderador no numérico: {eph['PONDERA'].isna().sum()}")

Registros EPH 2023T4 :    47,337
Suma PONDERA         :   29,528,610
Año/trimestre        : [2023], T[4]
Ponderador no numérico: 0


## 2. Distribución por estado laboral (formalidad para ocupados)

In [3]:
def estado_laboral(fila):
    est = fila["ESTADO"]
    if est == 1:
        emp = fila["EMPLEO"]
        if emp == 1:
            return "Empleo Formal"
        if emp == 2:
            return "Empleo Informal"
        return "Ocupado sin formalidad"
    if est == 2:
        return "Desocupación"
    if est == 3:
        return "Inactividad"
    if est == 4:
        if (fila["CH06"] >= 0) and (fila["CH06"] < 10):
            return "Menor de 10 años"
        return "Menor de 10 (edad inválida)"
    if est == 0:
        return "Entrevista no realizada"
    return "Otro/indefinido"

eph["estado_lab"] = eph.apply(estado_laboral, axis=1)

resumen_estados = (
    eph.groupby("estado_lab", dropna=False)
    .agg(registros=("PONDERA", "size"), poblacion=("PONDERA", "sum"))
    .sort_values("poblacion", ascending=False)
)
resumen_estados

,registros,poblacion
estado_lab,,
Inactividad,19078,11014283
Empleo Formal,12151,7893943
Empleo Informal,9114,5594996
Menor de 10 años,5467,3842677
Desocupación,1055,822050
Menor de 10 (edad inválida),351,258862
Entrevista no realizada,74,72801
Ocupado sin formalidad,47,28998


## 3. Los cuatro estados laborales y su concordancia con el total

Los cuatro estados del ABM son: **Empleo Formal, Empleo Informal, Desocupación e Inactividad** (ESTADO ∈ {1, 2, 3}).

In [4]:
masc_cuatro = eph["ESTADO"].isin([1, 2, 3])
cuatro_reg = int(masc_cuatro.sum())
cuatro_pob = eph.loc[masc_cuatro, "PONDERA"].sum()

print(f"Suma 4 estados     : {cuatro_reg:>9,} registros | {cuatro_pob:>12,.0f} personas")
print(f"Total EPH          : {total_reg:>9,} registros | {total_pob:>12,.0f} personas")
print(f"Fuera de 4 estados : {total_reg - cuatro_reg:>9,} registros | {total_pob - cuatro_pob:>12,.0f} personas")

print("\n== Quién queda fuera de los 4 estados ==")
print(eph.loc[~masc_cuatro]
      .groupby("estado_lab")
      .agg(registros=("PONDERA", "size"), poblacion=("PONDERA", "sum"))
      .to_string())

Suma 4 estados     :    41,445 registros |   25,354,270 personas
Total EPH          :    47,337 registros |   29,528,610 personas
Fuera de 4 estados :     5,892 registros |    4,174,340 personas

== Quién queda fuera de los 4 estados ==
                             registros  poblacion
estado_lab                                       
Entrevista no realizada             74      72801
Menor de 10 (edad inválida)        351     258862
Menor de 10 años                  5467    3842677


## 4. Población inicial del ABM

In [5]:
abm = pd.read_csv(ABM_CSV)
abm_reg = len(abm)
abm_pob = abm["peso"].astype(float).sum()

print(f"Agentes ABM    : {abm_reg:>9,}")
print(f"Suma peso      : {abm_pob:>12,.0f}")
print(f"Peso NaN       : {abm['peso'].isna().sum()}")
print(f"Año/trimestre  : {abm['ano'].unique()}, T{abm['trimestre'].unique()}")
print(f"id_persona duplicados: {abm['id_persona'].duplicated().sum()}")
print(f"id_agente duplicados : {abm['id_agente'].duplicated().sum()}")

print("\n== Estados del ABM ==")
print(abm.groupby("estado_actual")
      .agg(agentes=("id_agente", "size"), poblacion=("peso", "sum"))
      .to_string())

Agentes ABM    :    41,398
Suma peso      :   25,325,272
Peso NaN       : 0
Año/trimestre  : [2023], T[4]
id_persona duplicados: 0
id_agente duplicados : 0

== Estados del ABM ==
               agentes  poblacion
estado_actual                    
desocupacion      1055     822050
formal           12151    7893943
inactividad      19078   11014283
informal          9114    5594996


## 5. Comparación de universos y pérdida por paso

In [6]:
tabla_final = pd.DataFrame([
    {"Universo": "EPH completa 2023T4", "Registros": total_reg, "Población ponderada": total_pob},
    {"Universo": "Cuatro estados laborales (ESTADO 1/2/3)", "Registros": cuatro_reg, "Población ponderada": cuatro_pob},
    {"Universo": "ABM exportado", "Registros": abm_reg, "Población ponderada": abm_pob},
])
tabla_final

,Universo,Registros,Población ponderada
0,EPH completa 2023T4,47337,29528610.0
1,Cuatro estados laborales (ESTADO 1/2/3),41445,25354270.0
2,ABM exportado,41398,25325272.0


In [7]:
perdida_1 = total_pob - cuatro_pob
perdida_2 = cuatro_pob - abm_pob
perdida_total = total_pob - abm_pob

print("== PÉRDIDA DE PESO POBLACIONAL POR PASO ==")
print(f"EPH completa -> 4 estados : -{perdida_1:,.0f} personas ({100*perdida_1/total_pob:.2f}% del total EPH)")
print(f"4 estados -> ABM          : -{perdida_2:,.0f} personas ({100*perdida_2/total_pob:.2f}% del total EPH)")
print(f"Total                     : -{perdida_total:,.0f} personas ({100*perdida_total/total_pob:.2f}% del total EPH)")

== PÉRDIDA DE PESO POBLACIONAL POR PASO ==
EPH completa -> 4 estados : -4,174,340 personas (14.14% del total EPH)
4 estados -> ABM          : -28,998 personas (0.10% del total EPH)
Total                     : -4,203,338 personas (14.23% del total EPH)


## 6. Causas de exclusión

In [8]:
excluidos = eph[~eph["estado_lab"].isin(
    ["Empleo Formal", "Empleo Informal", "Desocupación", "Inactividad"])]

causas = (
    excluidos.groupby("estado_lab")
    .agg(registros=("PONDERA", "size"), poblacion=("PONDERA", "sum"))
    .assign(pct_total=lambda d: 100 * d["poblacion"] / total_pob)
    .sort_values("poblacion", ascending=False)
)
causas

,registros,poblacion,pct_total
estado_lab,,,
Menor de 10 años,5467,3842677,13.013403
Menor de 10 (edad inválida),351,258862,0.876648
Entrevista no realizada,74,72801,0.246544
Ocupado sin formalidad,47,28998,0.098203


## 7. Diagnóstico final

**¿Por qué el ABM representa ~25,3 M y no los ~30 M de INDEC?**

1. **EPH completa 2023T4** = 29.528.610 personas ponderadas.
2. **Cuatro estados laborales** (ESTADO 1/2/3) = 25.354.270. Se pierden **4.174.340 (14,14%)**:
   - **Menores de 10 años** (ESTADO=4): 5.467 registros, **3.842.677** personas (13,01%).
   - **Menores de 10 con edad inválida**: 351 registros, **258.862** personas (0,88%).
   - **Entrevista no realizada** (ESTADO=0): 74 registros, **72.801** personas (0,25%).
3. **ABM exportado** = 25.325.272 personas. Se pierden otros **28.998 (0,10%)**:
   - **Ocupados sin clasificación de formalidad** (ESTADO=1, EMPLEO=9): 47 registros.

**Conclusión:** el ABM no representa a toda la población de los 31 aglomerados, sino al **universo de 10 años o más en condición de actividad definida** (ocupados con formalidad clasificada, desocupados e inactivos). El grueso de la diferencia (~99% de la pérdida) son los **menores de 10 años**; el resto son casos con estado faltante o no clasificable. Esto es una decisión de diseño del ABM (modela el mercado laboral de la población en edad de actividad), no un error de muestreo.

In [9]:
# Guardar resultados para trazabilidad
OUT = RAIZ + r"\Salidas\13_abm_base\auditoria_universo_poblacional_abm"
tabla_final.to_csv(OUT + "_tabla_final.csv", index=False, encoding="utf-8-sig")
resumen_estados.to_csv(OUT + "_resumen_estados.csv", encoding="utf-8-sig")
causas.to_csv(OUT + "_causas_exclusion.csv", encoding="utf-8-sig")
print("CSV guardados en Salidas/13_abm_base/")

CSV guardados en Salidas/13_abm_base/
